# ICMP Echo Request Payload Analysis

In [ ]:
import pandas as pd
import ipaddress
import matplotlib.pyplot as plt
import os
import glob
import json

Output variables for input and output directories

In [ ]:
dir_input = "/Users/kix/src/ibram/08_ibrflow/test-nout4"
file_mask = "darknet_*_mod_non_fragmented_1_echoscan-ipid_echoscan-noipid.csv"

## Enrichement functions for IBR ICMP stats dataset

These functions include some columns to the dataframe. These coloumns will be removed
and the final information is moved to the IBR info dictionary column.

In [ ]:
# This function converts the ip_src or ip_dst from ipaddress to hexadecimal
def ip_to_hex(ip_str):
    ip = ipaddress.ip_address(ip_str)
    return ip.packed.hex()

In [ ]:
def include_ip_hex_columns(df):
    df['ip_src_hex'] = df['ip_src'].apply(ip_to_hex)
    df['ip_dst_hex'] = df['ip_dst'].apply(ip_to_hex)
    return df

In [ ]:
def include_flag_address_in_payload_columns(df):
    # Create two boolean colums with checking the payload_hex includes the hex representation of the IP addresses
    df['ip_src_in_payload'] = df.apply(lambda row: row['ip_src_hex'] in row['payload_hex'], axis=1)
    df['ip_dst_in_payload'] = df.apply(lambda row: row['ip_dst_hex'] in row['payload_hex'], axis=1)
    return df

In [ ]:
def print_ip_address_in_payload_stats(filename, df):
    # Count the number of ocurrences where ip_src or ip_dst is found in the payload
    src_count = df['ip_src_in_payload'].sum()
    dst_count = df['ip_dst_in_payload'].sum()

    # Count the number of ocurrences where the ip_src and the ip_dst are both found in the payload
    both_count = df[(df['ip_src_in_payload']) & (df['ip_dst_in_payload'])].shape[0]

    print(f"Stats for file {filename}: Total: {df.shape[0]}. ip_src in payload: {src_count}. ip_dst in payload: {dst_count}. Both in payload: {both_count}.")

    # print(f"Number of payloads containing ip_src: {src_count}")
    # print(f"Number of payloads containing ip_dst: {dst_count}")
    # print(f"Number of payloads containing both ip_src and ip_dst: {both_count}")
    # print(f"Total number of payloads: {df.shape[0]}")

In [ ]:
def include_payload_length_column(df):
    # Include a new columen with the length of the payload in bytes
    df['payload_length_bytes'] = df['payload_hex'].apply(lambda x: len(x) // 2)
    return df

In [ ]:
def copy_info_to_ibrjson(df):
    # Copy the enrichment information to the IBR info dictionary column
    def update_ibr_info(row):
        ibr_info = row.get('ibr_info', {})
        ibr_info['ip_src_hex'] = row['ip_src_hex']
        ibr_info['ip_dst_hex'] = row['ip_dst_hex']
        ibr_info['ip_src_in_payload'] = row['ip_src_in_payload']
        ibr_info['ip_dst_in_payload'] = row['ip_dst_in_payload']
        ibr_info['payload_length_bytes'] = row['payload_length_bytes']
        return ibr_info

    df['ibr_info'] = df.apply(update_ibr_info, axis=1)
    return df

## Load the dataset
This dataset was generated after using the scripts ibrflow_agg_icmpscan_ipid.py and ibrflow_agg_icmpscan_noipid.py and merging the results in the CSV files.
The dataset is created using the files in the specified input directory.

In [ ]:
all_files = glob.glob(os.path.join(dir_input, file_mask))

In [ ]:
columns_out_icmp = [
    "timestamp", "ip_version", "ip_orig_tos", "ip_tos", "ip_prec", "ip_dscp",
    "ip_enc", "ip_len", "ip_id", "ip_ttl", "ip_chksum", "ip_src", "ip_dst",
    "ip_options", "icmp_type", "icmp_code", "icmp_id", "icmp_seq",
    "icmp_chksum", "payload_hex", "key", "IBR_info",
]

In [ ]:
def load_and_enrich_data(all_files):
    df_full = pd.DataFrame()

    # Use a loop to read, process, enrich the dataframe
    for file in all_files:
        print(f"Processing file: {file}")
        if file.endswith('.csv'):
            df = pd.read_csv(file, sep=';', names=columns_out_icmp)
        elif file.endswith('.csv.gz'):
            df = pd.read_csv(file, compression='gzip', sep=';', names=columns_out_icmp)
        else:
            print(f"Unsupported file format: {file}")
            continue

        # Convert the IBR_info column from JSON string to actual dictionary
        df['IBR_info'] = df['IBR_info'].apply(json.loads)

        # Update the dataframe with enrichment functions
        df = include_ip_hex_columns(df)
        df = include_flag_address_in_payload_columns(df)
        df = include_payload_length_column(df)
        print_ip_address_in_payload_stats(file, df)
        df = copy_info_to_ibrjson(df)
        # df = remove_temp_columns(df)
        df_full = pd.concat([df_full, df], ignore_index=True)

    return df_full

In [ ]:
df = load_and_enrich_data(all_files)

In [ ]:
df.columns

In [ ]:
src_count = df['ip_src_in_payload'].sum()
dst_count = df['ip_dst_in_payload'].sum()
both_count = df[(df['ip_src_in_payload']) & (df['ip_dst_in_payload'])].shape[0]

In [ ]:
print(f"Number of payloads containing ip_src: {src_count}")
print(f"Number of payloads containing ip_dst: {dst_count}")
print(f"Number of payloads containing both ip_src and ip_dst: {both_count}")
print(f"Total number of payloads: {df.shape[0]}")

In [ ]:
# Aggregate by payload length and ip_src_in_payload
agg_counts = df.groupby(['payload_length_bytes', 'ip_dst_in_payload']).size().unstack(fill_value=0)
agg_counts.plot(kind='bar', stacked=True, title='Payload Length vs IP Destination Inclusion', xlabel='Payload Length (bytes)', ylabel='Number of Occurrences')

In [ ]:
# Aggregate by payload length, counting unique ip_src for each condition
agg_src_in = df.groupby(['payload_length_bytes', 'ip_src_in_payload'])['ip_src'].nunique().unstack(fill_value=0)
agg_dst_in = df.groupby(['payload_length_bytes', 'ip_dst_in_payload'])['ip_src'].nunique().unstack(fill_value=0)

# Rename columns for clarity
agg_src_in.columns = ['ip_src NOT in payload', 'ip_src IN payload']
agg_dst_in.columns = ['ip_dst NOT in payload', 'ip_dst IN payload']

# Combine into a single DataFrame
combined = pd.concat([agg_src_in, agg_dst_in], axis=1)

# Plot
ax = combined.plot(kind='bar', figsize=(12, 6), width=0.8)
ax.set_title('Payload Length vs IP Address Inclusion (by unique IP sources)')
ax.set_xlabel('Payload Length (bytes)')
ax.set_ylabel('Number of Unique IP Sources')
ax.legend(title='Condition')
plt.tight_layout()

In [ ]:
# Get the lengths with payload that includes the ip_src or ip_dst
lengths_with_ip_src = df[df['ip_src_in_payload']]['payload_length_bytes'].unique()
lengths_with_ip_dst = df[df['ip_dst_in_payload']]['payload_length_bytes'].unique()

# Convert the np.int64 arrays to regular Python lists for better printing
lengths_with_ip_src = [int(length) for length in lengths_with_ip_src]
lengths_with_ip_dst = [int(length) for length in lengths_with_ip_dst]

print(f"Payload lengths with ip_src included: {sorted(lengths_with_ip_src)}")
print(f"Payload lengths with ip_dst included: {sorted(lengths_with_ip_dst)}")

In [ ]:
# Create a combined category column for stacked bar chart
def categorize_payload(row):
    if row['ip_src_in_payload'] and row['ip_dst_in_payload']:
        return 'Both in payload'
    elif row['ip_src_in_payload']:
        return 'Source IP in payload'
    elif row['ip_dst_in_payload']:
        return 'Destination IP in payload'
    else:
        return 'Neither in payload'

df['ip_in_payload_category'] = df.apply(categorize_payload, axis=1)

# Aggregate by payload length and category, counting unique ip_src
agg_category = df.groupby(['payload_length_bytes', 'ip_in_payload_category'])['ip_src'].nunique().unstack(fill_value=0)

# Reorder columns for better visualization
col_order = ['Destination IP in payload', 'Source IP in payload', 'Neither in payload', 'Both in payload']
agg_category = agg_category.reindex(columns=[c for c in col_order if c in agg_category.columns], fill_value=0)

# Font size configuration
font_size = 22

# Plot stacked bar chart
ax = agg_category.plot(kind='bar', stacked=True, figsize=(12, 6), colormap='Set1')
# ax.set_title('Payload Length vs IP Address Inclusion (by unique IP sources)', fontsize=font_size + 2)
ax.set_xlabel('Payload Length (bytes)', fontsize=font_size, labelpad=15)
ax.set_ylabel('Number of Unique IP Sources', fontsize=font_size)
ax.legend(loc='upper right', fontsize=font_size - 2, title_fontsize=font_size)
ax.tick_params(axis='both', labelsize=font_size - 2)
plt.tight_layout()
plt.savefig('icmp_payload_length_ip_inclusion.pdf', dpi=300)

In [ ]:
# For these lengths, show some example payloads with the ip_src or ip_dst included

# Examples of payloads for specific lengths with ip_src included
for length in sorted(lengths_with_ip_src):
    sample_payloads_src = df[(df['payload_length_bytes'] == length) & (df['ip_src_in_payload'])].head(5)
    if not sample_payloads_src.empty:
        print(f"Sample payloads with length {length} bytes including ip_src:")
        print(sample_payloads_src[['ip_src_hex', 'payload_hex']].to_string(index=False))
        print("\n")

# Examples of payloads for specific lengths with ip_dst included
for length in sorted(lengths_with_ip_dst):
    sample_payloads_dst = df[(df['payload_length_bytes'] == length) & (df['ip_dst_in_payload'])].head(5)
    if not sample_payloads_dst.empty:
        print(f"Sample payloads with length {length} bytes including ip_dst:")
        print(sample_payloads_dst[['ip_dst_hex', 'payload_hex']].to_string(index=False))
        print("\n")

In [ ]:
### Examples of payloads for specific lengths
lengths_to_check = [8, 16, 20, 32, 40, 56, 64, 71, 72]
for length in lengths_to_check:
    sample_payloads = df[df['payload_length_bytes'] == length].head(5)
    print(f"Sample payloads with length {length} bytes:")
    print(sample_payloads[['payload_hex']].to_string(index=False))
    print("\n")

In [ ]:
# Include a boolean field to check if the data in payload is structured, using the string 0000 as indicator
def is_structured_payload(payload_hex):
    return '0000' in payload_hex

df['structured_payload'] = df['payload_hex'].apply(is_structured_payload)

In [ ]:
df['structured_payload'].value_counts()

In [ ]:
# Group by structured_payload, payload_length_bytes, ip_src, ip_src_in_payload, ip_dst_in_payload
grouped = df.groupby(['structured_payload', 'payload_length_bytes', 'ip_src', 'ip_src_in_payload', 'ip_dst_in_payload']).size().reset_index(name='count')


In [ ]:
grouped.sort_values(by='count', ascending=False)

In [ ]:
grouped[(grouped['count'] > 1)]

In [ ]:
grouped[(grouped['payload_length_bytes'] == 20) & (grouped['count'] == 1)]['ip_src'].to_list()

In [ ]:
# Group by structured_payload, payload_length_bytes, ip_src_in_payload, ip_dst_in_payload
grouped_no_ip_src = df.groupby(['structured_payload', 'payload_length_bytes', 'ip_src_in_payload', 'ip_dst_in_payload']).size().reset_index(name='count')

In [ ]:
grouped_no_ip_src[grouped_no_ip_src['count'] > 1]